# 09.2 — Native Deep Agent Durability and Interrupts

## Goal

Prove durable execution and human-in-the-loop (HITL) **directly on the existing
LangChain Deep Agents research harness** — without wrapping it in a separate
outer StateGraph workflow.

This continues from 09.1 but takes the opposite structural choice: for an
open-ended, ChatGPT-like research agent, Deep Agents stays the primary
orchestration layer and the LangGraph runtime underneath owns durable state.

## Architecture

```
Deep Agent            <- primary orchestration (planning, tools, subagents)
  |
LangGraph runtime     <- durable execution + interrupts
  |
thread_id + checkpointer
  |
SQLite (dev)  ->  Postgres (later)
```

Key identity rule:

- **LangGraph `thread_id`** = durable application thread identity.
- **Foundry `session_id`** = runtime/workspace lifecycle only; never reused as
  the permanent `thread_id`, and Foundry Responses history is **not** hydrated
  into LangGraph.

## Prerequisites

- `az login` completed; `AZURE_AI_PROJECT_ENDPOINT` and
  `AZURE_AI_MODEL_DEPLOYMENT_NAME` set in `.env`.
- This notebook makes **live Foundry calls** for the research turns.
- SQLite is **development-only** persistence. Postgres is a later slice.

In [1]:
from langgraph.types import Command

from deep_agents_foundry import (
    build_research_agent,
    build_sqlite_checkpointer,
    thread_config,
    content_text,
)

## 1. The Deep Agent without persistence

By default the agent is stateless: no `thread_id`, no checkpointer. Each call
starts fresh with no memory of previous turns.

In [2]:
stateless_agent = build_research_agent()

result = stateless_agent.invoke(
    {'messages': [{'role': 'user', 'content': 'In one sentence, what is a Microsoft Foundry Hosted Agent?'}]}
)

print(content_text(result['messages'][-1]))

C:\Users\shchitt\Downloads\Projects\deep-agents-on-foundry\src\deep_agents_foundry\tools.py:13: ExperimentalWarning: WebSearchTool is currently in preview and is subject to change. This preview is provided without a service-level agreement, and we don't recommend it for production workloads. Certain features might not be supported or might have constrained capabilities. For more information, see https://azure.microsoft.com/support/legal/preview-supplemental-terms
  return WebSearchTool()


A **Microsoft Foundry Hosted Agent** is an AI agent you package as your own code (typically containerized) and deploy to **Microsoft-managed Foundry Agent Service** infrastructure, which handles runtime concerns like scaling, security/identity, and session state so you can focus on the agent’s logic. ([learn.microsoft.com](https://learn.microsoft.com/en-us/agent-framework/hosting/foundry-hosted-agent))


## 2. Add a SQLite checkpointer

`build_sqlite_checkpointer()` creates the parent directory, opens a `sqlite3`
connection with `check_same_thread=False`, and returns a LangGraph `SqliteSaver`
— development persistence only.

In [3]:
checkpointer = build_sqlite_checkpointer('../data/checkpoints.db')
print(type(checkpointer))

<class 'langgraph.checkpoint.sqlite.SqliteSaver'>


## 3. Build the research Deep Agent with the checkpointer

Same harness as before — we only inject persistence via the existing
`checkpointer` seam.

In [4]:
agent = build_research_agent(checkpointer=checkpointer)
print(type(agent))

<class 'langgraph.graph.state.CompiledStateGraph'>


C:\Users\shchitt\Downloads\Projects\deep-agents-on-foundry\src\deep_agents_foundry\tools.py:13: ExperimentalWarning: WebSearchTool is currently in preview and is subject to change. This preview is provided without a service-level agreement, and we don't recommend it for production workloads. Certain features might not be supported or might have constrained capabilities. For more information, see https://azure.microsoft.com/support/legal/preview-supplemental-terms
  return WebSearchTool()


## 4. Run a real web-research turn under a stable thread_id

The `thread_id` is the durable application thread identity. The application owns
it — we do not derive it from a Foundry session.

In [5]:
config = thread_config('research-thread-durable-001')

first = agent.invoke(
    {'messages': [{'role': 'user', 'content': 'Research the key operational responsibilities Microsoft Foundry manages for Hosted Agents. Give a concise summary with citations.'}]},
    config=config,
)

print(content_text(first['messages'][-1]))

Microsoft Foundry (Foundry Agent Service) takes on the **platform/ops work around your code** for **Hosted Agents**—you provide the containerized agent code, while Foundry operates the runtime and surrounding operational concerns:

- **Managed endpoint + request routing:** When you deploy, Foundry **exposes a dedicated endpoint** for the agent and routes runtime requests to your container. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/hosted-agents))  
- **Identity management (agent identity):** Foundry **assigns a dedicated Microsoft Entra ID (“agent identity”)** to the deployed agent so it can authenticate to models, tools, and downstream Azure services. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/hosted-agents))  
- **Autoscaling / scale-to-zero:** Foundry “**hosts and scales**” hosted agents, and the hosted-agent runtime is described as supporting **scale-to-zero economics** (scaling down when idle)

## 5. Follow-up without resending history

We send only the new question. LangGraph restores the prior messages from the
checkpoint under the same `thread_id`.

In [6]:
followup = agent.invoke(
    {'messages': [{'role': 'user', 'content': 'Now expand specifically on the identity and RBAC responsibilities.'}]},
    config=config,
)

print(content_text(followup['messages'][-1]))

### Identity (Microsoft Entra) — who does what

- **Microsoft Foundry provisions the runtime identity for the agent.** Each Hosted agent gets an **instance identity (an Entra ID service principal)** that the agent uses at runtime to authenticate to downstream resources. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/manage-hosted-agent))  
- **You (or your platform admin) decide what that identity can access.** To let the agent reach services like **Azure Storage / Cosmos DB**, you must take the identity’s **principal ID** and then grant access via RBAC role assignments at the appropriate scope. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/manage-hosted-agent))  
- **Purpose of agent identities:** Foundry’s agent identity model is explicitly intended to support (a) **management/governance** (inventory, policy, audit) and (b) **tool authentication without embedding secrets** in prompts/code. ([learn.microsoft.

In [7]:
state = agent.get_state(config)
messages = state.values['messages']

print('Total messages on thread:', len(messages))
for i, message in enumerate(messages):
    print(i, type(message).__name__)

Total messages on thread: 4
0 HumanMessage
1 AIMessage
2 HumanMessage
3 AIMessage


The message count includes **both** turns even though the second call sent only
one new message. State lives in the checkpoint, not in the request payload.

## 6. Reconstruct the agent against the same SQLite DB

Discard the agent object and build a brand-new one pointing at the **same**
database file. The thread still exists.

In [8]:
del agent

restored_checkpointer = build_sqlite_checkpointer('../data/checkpoints.db')
restored_agent = build_research_agent(checkpointer=restored_checkpointer)

restored_state = restored_agent.get_state(config)
print('Restored messages on thread:', len(restored_state.values['messages']))

Restored messages on thread: 4


C:\Users\shchitt\Downloads\Projects\deep-agents-on-foundry\src\deep_agents_foundry\tools.py:13: ExperimentalWarning: WebSearchTool is currently in preview and is subject to change. This preview is provided without a service-level agreement, and we don't recommend it for production workloads. Certain features might not be supported or might have constrained capabilities. For more information, see https://azure.microsoft.com/support/legal/preview-supplemental-terms
  return WebSearchTool()


Durability belongs to the **LangGraph checkpoint**, not to Python process
memory. A new object on the same DB sees the same thread.

## 7. Inspect the installed Deep Agents API (native HITL)

Use the mechanism the pinned version actually supports rather than guessing.

In [9]:
import inspect
import deepagents

print('deepagents version:', deepagents.__version__)

params = inspect.signature(deepagents.create_deep_agent).parameters
print('interrupt_on:', params['interrupt_on'].annotation)
print('checkpointer:', params['checkpointer'].annotation)

from langchain.agents.middleware.human_in_the_loop import (
    HumanInTheLoopMiddleware,
    InterruptOnConfig,
)
print('HITL middleware:', HumanInTheLoopMiddleware.__name__)

deepagents version: 0.7.11
interrupt_on: dict[str, bool | langchain.agents.middleware.human_in_the_loop.InterruptOnConfig] | None
checkpointer: None | bool | langgraph.checkpoint.base.BaseCheckpointSaver
HITL middleware: HumanInTheLoopMiddleware


### Native HITL mechanism

`create_deep_agent(..., interrupt_on={tool_name: True})` attaches the built-in
`HumanInTheLoopMiddleware`. When a gated tool is about to run, the graph
**interrupts** and checkpoints the paused state.

Resume with LangGraph's native command:

```
Command(resume={'decisions': [{'type': 'approve'}]})
```

Allowed decision types include `approve`, `reject`, and `edit`. We do **not**
build a custom middleware — this is the harness's built-in mechanism.

## 8. Add a harmless approval-gated tool (notebook-only)

A local, side-effect-light tool to demonstrate approval. It is **notebook-only**
and intentionally not part of the production research toolset.

In [10]:
from langchain_core.tools import tool

_saved_notes = {}

@tool
def save_research_summary(title: str, summary: str) -> str:
    '''Save a research summary. Notebook-only approval-gated demo tool.'''
    _saved_notes[title] = summary
    return f'Saved research summary {title!r} ({len(summary)} chars).'

## 9. Configure a Deep Agent so the tool triggers a native interrupt

Here we call `create_deep_agent` directly because we need to add the extra local
tool **and** gate it. This is still the same Deep Agents harness — same model,
instructions, and web search — plus one approval-gated tool. `build_research_agent`
remains the stateless/production path.

In [11]:
from deepagents import create_deep_agent

from deep_agents_foundry.model import build_model
from deep_agents_foundry.tools import build_web_search_tool
from deep_agents_foundry.agent import RESEARCH_INSTRUCTIONS

hitl_checkpointer = build_sqlite_checkpointer('../data/native_hitl.db')

hitl_agent = create_deep_agent(
    model=build_model(),
    tools=[build_web_search_tool(), save_research_summary],
    system_prompt=RESEARCH_INSTRUCTIONS,
    interrupt_on={'save_research_summary': True},
    checkpointer=hitl_checkpointer,
)
print(type(hitl_agent))

<class 'langgraph.graph.state.CompiledStateGraph'>


C:\Users\shchitt\Downloads\Projects\deep-agents-on-foundry\src\deep_agents_foundry\tools.py:13: ExperimentalWarning: WebSearchTool is currently in preview and is subject to change. This preview is provided without a service-level agreement, and we don't recommend it for production workloads. Certain features might not be supported or might have constrained capabilities. For more information, see https://azure.microsoft.com/support/legal/preview-supplemental-terms
  return WebSearchTool()


## 10. Run until the interrupt

We explicitly ask the agent to save a summary. When it calls
`save_research_summary`, execution pauses **before** the tool runs.

In [12]:
hitl_config = thread_config('research-hitl-001')

interrupted = hitl_agent.invoke(
    {'messages': [{'role': 'user', 'content': 'Briefly research Foundry Hosted Agent identity, then save your summary using the save_research_summary tool with a short title.'}]},
    config=hitl_config,
)

print('Interrupted:', '__interrupt__' in interrupted)
print(interrupted.get('__interrupt__'))

Interrupted: True
[Interrupt(value={'action_requests': [{'name': 'save_research_summary', 'args': {'title': 'Foundry Hosted Agent identity (quick summary)', 'summary': 'In Microsoft Foundry Agent Service, each deployed Hosted Agent is automatically provisioned with its own dedicated Microsoft Entra ID–backed “agent identity” (created at deploy time) that the platform uses for governance/auditing and for authenticating the agent’s tool calls to downstream services without embedding secrets (tokens are requested on demand and access is controlled via least-privilege RBAC assignments to the agent identity). \ue200cite\ue202turn0search0\ue202turn0search7\ue202turn0search9\ue201'}, 'description': "Tool execution requires approval\n\nTool: save_research_summary\nArgs: {'title': 'Foundry Hosted Agent identity (quick summary)', 'summary': 'In Microsoft Foundry Agent Service, each deployed Hosted Agent is automatically provisioned with its own dedicated Microsoft Entra ID–backed “agent identity

## 10b. Inspect the interrupted state/checkpoint

In [13]:
state = hitl_agent.get_state(hitl_config)

print('Next steps pending:', state.next)
print('Saved notes so far:', dict(_saved_notes))

Next steps pending: ('HumanInTheLoopMiddleware.after_model',)
Saved notes so far: {}


`state.next` is non-empty (work remains) and `_saved_notes` is still empty — the
tool has **not** executed. The pause is checkpointed in SQLite.

## 11. Resume using native Command(resume=...)

In [14]:
resumed = hitl_agent.invoke(
    Command(resume={'decisions': [{'type': 'approve'}]}),
    config=hitl_config,
)

print(content_text(resumed['messages'][-1]))
print('Saved notes now:', dict(_saved_notes))

I researched “Foundry Hosted Agent identity” in **Microsoft Foundry Agent Service**:

- A **Hosted Agent** gets a **dedicated agent identity in Microsoft Entra ID** that is **automatically created at deploy time** (along with a dedicated endpoint).   
- This **agent identity** is used to **authenticate the agent to downstream tools/services** and enables **governance/auditing and least-privilege access via RBAC**, reducing the need to embed long-lived secrets in agent code.   

Saved via `save_research_summary` as: **“Foundry Hosted Agent identity (quick summary)”**.
Saved notes now: {'Foundry Hosted Agent identity (quick summary)': 'In Microsoft Foundry Agent Service, each deployed Hosted Agent is automatically provisioned with its own dedicated Microsoft Entra ID–backed “agent identity” (created at deploy time) that the platform uses for governance/auditing and for authenticating the agent’s tool calls to downstream services without embedding secrets (tokens are requested on demand a

## 12. Prove continuation, not restart

In [15]:
from langchain_core.messages import ToolMessage

state_after = hitl_agent.get_state(hitl_config)
print('Next steps pending:', state_after.next)  # () means complete

tool_runs = [
    m for m in state_after.values['messages']
    if isinstance(m, ToolMessage) and 'Saved research summary' in str(m.content)
]
print('save_research_summary executions:', len(tool_runs))

Next steps pending: ()
save_research_summary executions: 1


The research done before the interrupt is preserved, the tool executed **once**,
and the run completed from the checkpoint — it did not restart the whole task.

## 13. Resume an interrupted thread after process/agent reconstruction

Start a fresh interrupted thread, discard the agent, rebuild against the same DB,
and resume — proving the interrupt lives in the checkpoint, not memory.

In [16]:
restart_config = thread_config('research-hitl-restart-001')

hitl_agent.invoke(
    {'messages': [{'role': 'user', 'content': 'Briefly note one Foundry Hosted Agent benefit, then save it using the save_research_summary tool.'}]},
    config=restart_config,
)

print('Interrupted thread next:', hitl_agent.get_state(restart_config).next)

Interrupted thread next: ('HumanInTheLoopMiddleware.after_model',)


In [17]:
del hitl_agent

rebuilt_checkpointer = build_sqlite_checkpointer('../data/native_hitl.db')
rebuilt_hitl_agent = create_deep_agent(
    model=build_model(),
    tools=[build_web_search_tool(), save_research_summary],
    system_prompt=RESEARCH_INSTRUCTIONS,
    interrupt_on={'save_research_summary': True},
    checkpointer=rebuilt_checkpointer,
)

print('Pending before resume:', rebuilt_hitl_agent.get_state(restart_config).next)

rebuilt_resumed = rebuilt_hitl_agent.invoke(
    Command(resume={'decisions': [{'type': 'approve'}]}),
    config=restart_config,
)

print(content_text(rebuilt_resumed['messages'][-1]))

C:\Users\shchitt\Downloads\Projects\deep-agents-on-foundry\src\deep_agents_foundry\tools.py:13: ExperimentalWarning: WebSearchTool is currently in preview and is subject to change. This preview is provided without a service-level agreement, and we don't recommend it for production workloads. Certain features might not be supported or might have constrained capabilities. For more information, see https://azure.microsoft.com/support/legal/preview-supplemental-terms
  return WebSearchTool()


Pending before resume: ('HumanInTheLoopMiddleware.after_model',)
One Foundry Hosted Agent benefit: **reduced operational overhead**—the platform hosts and manages the agent runtime (deployment/scaling/maintenance), letting teams focus on agent logic.

Saved via `save_research_summary` as **“Foundry Hosted Agent benefit.”**


A brand-new agent object resumed a thread that a **different** object had
interrupted — the interrupt/resume state lives in the checkpoint, not memory.

## 14. Durable state vs HITL vs retry vs restart vs background

- **Durable thread state** — graph state (messages, checkpoints, pending work)
  persists across invocations and process restarts.
- **Human-in-the-loop interrupt** — execution intentionally pauses before a
  gated action and waits for an external decision.
- **Retry** — run the *failed step* again.
- **Restart** — start the *whole task* from the beginning (loses progress).
- **Background execution** — the run continues without the client connection
  staying open. This is orthogonal: durability is about *surviving* pauses and
  restarts, not about who is watching.

## 15. Final architecture

```
Client
  | Responses
Foundry Hosted Agent
  |
Deep Agent
  |-- planning
  |-- filesystem
  |-- web search
  |-- subagents
  |-- skills
  +-- HITL
  |
LangGraph runtime
  |
thread_id + checkpointer
  |
SQLite now / Postgres later
```

## When should I use an outer LangGraph workflow instead?

Use the **Deep Agent harness directly** (this notebook) for open-ended,
general-purpose research. The agent decides its own steps; durability and HITL
come from the LangGraph runtime beneath it.

Reach for a **separate outer LangGraph `StateGraph`** only when you have a
**fixed business process** with named, ordered stages and explicit gates that
must hold regardless of model judgment — for example a compliance pipeline like
`intake -> mandatory legal approval -> publish`.

Rule of thumb:

- Dynamic reasoning / open-ended research -> Deep Agent harness.
- Fixed, auditable, non-negotiable stage order -> outer StateGraph.

For our ChatGPT-like research product, the Deep Agent harness is the right
default; 09.1's outer workflow was a teaching example, not the product shape.

## Summary

- **Native HITL API used:** `create_deep_agent(..., interrupt_on={'tool': True})`
  (built-in `HumanInTheLoopMiddleware`), resumed with
  `Command(resume={'decisions': [{'type': 'approve'}]})`.
- **Durability proven:** a follow-up turn restored prior messages under the same
  `thread_id` without resending history.
- **Process restart proven:** rebuilding the agent against the same SQLite DB
  restored the thread, and a freshly reconstructed agent resumed a thread that
  another agent had interrupted.
- **Remaining for P7C productionization:** decide whether any approval-gated tool
  belongs in the production toolset, expose a minimal HITL/resume seam through
  `hosting.py` (P7D) mapping client thread identity -> `thread_id`, and swap the
  SQLite checkpointer for Postgres — without hydrating Foundry Responses history
  into LangGraph.